# OntoHajj – GPU timing run (Experiment B3)
Runs S4 and S2 with Qwen2.5-7B (same Ollama model as the laptop run) on the same stratified 100 test questions, on a free Colab GPU.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → Run all.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1) install and start Ollama, download the model (about 5 GB, 3-5 minutes)
!sudo apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)
!ollama pull qwen2.5:7b

In [ ]:
# 2) get the code and data
!git clone -q https://github.com/majidphd/OntoHajj
%cd OntoHajj
!pip -q install -r requirements.txt

In [ ]:
# 3) settings: local Ollama server, same as on the laptop
import os
os.environ.update({'ONTOHAJJ_PROVIDER': 'openai', 'ONTOHAJJ_BASE_URL': 'http://localhost:11434/v1',
                   'ONTOHAJJ_API_KEY': 'ollama', 'ONTOHAJJ_NO_THINK': '0'})
!python eval/test_connection.py qwen2.5:7b

In [ ]:
# 4) the run (about 20-40 minutes on a T4); results go to results/llm/qwen2.5_7b/test_gpu
!python eval/run_llm_eval.py --model qwen2.5:7b --split test --configs C4 C2 --limit 100 --version gpu

In [ ]:
# 5) summary: accuracy and latency on the GPU, and the laptop (CPU) results on the same 100 questions
import json, statistics
gpu = {c: [json.loads(l) for l in open(f'results/llm/qwen2.5_7b/test_gpu/{c}_run0.jsonl')] for c in ['C4', 'C2']}
ids = [r['id'] for r in gpu['C4']]
for c, rows in gpu.items():
    cpu = {json.loads(l)['id']: json.loads(l) for l in open(f'results/llm/qwen2.5_7b/test_v2/{c}_run0.jsonl')}
    lat = [r['latency'] for r in rows if r['latency'] is not None]
    llm = [r['latency'] for r in rows if r['latency'] is not None and not (r['reason'] or '').startswith(('L2', 'L4'))]
    print(f"{ {'C4':'S4','C2':'S2'}[c] }: GPU accuracy {sum(r['correct'] for r in rows)}/{len(rows)}, CPU accuracy on the same questions {sum(cpu[r['id']]['correct'] for r in rows)}/{len(rows)}")
    print(f"    GPU latency per question: mean {statistics.mean(lat):.2f} s, median {statistics.median(lat):.2f} s; questions that called the LLM: mean {statistics.mean(llm):.2f} s")
    print(f"    CPU latency on the same questions: mean {statistics.mean(cpu[r['id']]['latency'] for r in rows):.1f} s")

In [ ]:
# 6) download the results folder and send it
!cd results/llm/qwen2.5_7b && zip -qr /content/test_gpu.zip test_gpu
from google.colab import files
files.download('/content/test_gpu.zip')